# Calibration & Validation — Discharge Comparison Plot

**Purpose:** Produces a combined observed vs. simulated discharge plot that
covers both the calibration and validation periods, with a dual metrics box
showing NSE, RMSE, KGE, r, β, and γ for each period.

**What it does:**
- Implements NSE, RMSE, and KGE (with sub-components r, β, γ)
- Reads discharge data and separates it into calibration / validation windows
- Plots both periods on a shared time axis with a metrics overlay box
  labelled 'Kalibrierung' and 'Validierung'

**Input:** Merged observed + simulated discharge Excel file  
**Output:** Combined calibration–validation hydrograph with metrics

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.offsetbox import AnchoredOffsetbox, TextArea, HPacker, VPacker

### Calculation formula for different indices

In [ ]:
# 1. NSE
def nse(q_obs, q_sim):

    mask = np.isfinite(q_obs) & np.isfinite(q_sim)
    q_obs, q_sim = q_obs[mask], q_sim[mask]

    if len(q_obs) == 0 or np.all(q_obs == q_obs[0]):
        return np.nan

    numerator = np.sum((q_obs - q_sim) ** 2)
    denominator = np.sum((q_obs - np.mean(q_obs)) ** 2)

    return 1 - numerator / denominator

# 2. RMSE

def rmse(q_obs, q_sim):

    mask = np.isfinite(q_obs) & np.isfinite(q_sim)
    q_obs, q_sim = q_obs[mask], q_sim[mask]

    if len(q_obs) == 0:
        return np.nan

    return np.sqrt(np.mean((q_obs - q_sim) ** 2))


# 3. KGE

def kge(q_obs, q_sim):

    mask = np.isfinite(q_obs) & np.isfinite(q_sim)
    q_obs, q_sim = q_obs[mask], q_sim[mask]

    if len(q_obs) == 0:
        return np.nan, np.nan, np.nan, np.nan

    if np.std(q_obs) == 0 or np.std(q_sim) == 0:
        r = np.nan
    else:
        r = np.corrcoef(q_obs, q_sim)[0, 1]

    mean_obs, mean_sim = np.mean(q_obs), np.mean(q_sim)
    std_obs, std_sim = np.std(q_obs), np.std(q_sim)

    cv_obs = std_obs / mean_obs if mean_obs != 0 else np.nan
    cv_sim = std_sim / mean_sim if mean_sim != 0 else np.nan

    beta = mean_sim / mean_obs if mean_obs != 0 else np.nan
    gamma = cv_sim / cv_obs if cv_obs != 0 else np.nan

    if np.isnan(r) or np.isnan(beta) or np.isnan(gamma):
        return np.nan, r, beta, gamma

    kge_value = 1 - np.sqrt((r - 1) ** 2 + (gamma - 1) ** 2 + (beta - 1) ** 2)

    return kge_value, r, beta, gamma




### Metrics Box with different performance indices

In [ ]:

def add_metrics_box(ax,NSE_cal, RMSE_cal, KGE_cal, r_cal, beta_cal, gamma_cal,
                   NSE_val, RMSE_val, KGE_val, r_val, beta_val, gamma_val):
    cal_header = TextArea(
        "Kalibrierung",
        textprops=dict(weight='normal', size=8, family="monospace")
    )

    cal_body = TextArea(
        f"NSE : {NSE_cal:.3f}\n"
        f"RMSE : {RMSE_cal:.3f}\n"
        f"KGE : {KGE_cal:.3f}\n"
        f"r   : {r_cal:.3f}\n"
        f"β   : {beta_cal:.3f}\n"
        f"γ   : {gamma_cal:.3f}",
        textprops=dict(size=8, family="monospace")
    )

    left_col = VPacker(children=[cal_header, cal_body], align="left", pad=0, sep=3)


    val_header = TextArea(
        "Validierung",
        textprops=dict(weight='normal', size=8, family="monospace")
    )

    val_body = TextArea(
        f"NSE : {NSE_val:.3f}\n"
        f"RMSE : {RMSE_val:.3f}\n"
        f"KGE : {KGE_val:.3f}\n"
        f"r   : {r_val:.3f}\n"
        f"β   : {beta_val:.3f}\n"
        f"γ   : {gamma_val:.3f}",
        textprops=dict(size=8, family="monospace")
    )

    right_col = VPacker(children=[val_header, val_body], align="left", pad=0, sep=3)

    hbox = HPacker(children=[left_col, right_col], align="top", pad=0, sep=40)

    anchored = AnchoredOffsetbox(
        loc='upper left',
        child=hbox,
        frameon=True,
        borderpad=0.6
    )

    anchored.patch.set_boxstyle("round,pad=0.4")
    anchored.patch.set_facecolor("white")
    anchored.patch.set_alpha(0.6)
    anchored.patch.set_edgecolor("0.7")

    ax.add_artist(anchored)

### Function to define Calibration and Validation period

In [ ]:
def main():

    merged_file = r"C:\Users\raah\Desktop\Aufgabe\TALSIM_NG\EZG\Ziegenrueck\Simulation_ZR\Calibration.xlsx"

    df = pd.read_excel(merged_file)

    # Convert datetime
    df["DateTime"] = pd.to_datetime(df["Zeit"], format="%d.%m.%Y %H:%M", errors="coerce")

    df = df.dropna(subset=["DateTime", "Obs", "Sim"])



    # Calibration / Validation
    # -------------------------
    calib_start = pd.to_datetime("2011-11-01 00:00")
    calib_end   = pd.to_datetime("2017-10-31 23:00")

    valid_start = pd.to_datetime("2017-11-01 00:00")
    valid_end   = pd.to_datetime("2022-10-31 23:00")


    df = df[(df["DateTime"] >= calib_start) & (df["DateTime"] <= valid_end)]


    # -------------------------
    # Metrics
    # -------------------------
    df_cal = df[(df["DateTime"] >= calib_start) & (df["DateTime"] <= calib_end)]
    df_val = df[(df["DateTime"] >= valid_start) & (df["DateTime"] <= valid_end)]

    NSE_cal = nse(df_cal["Obs"].values, df_cal["Sim"].values)
    RMSE_cal = rmse(df_cal["Obs"].values, df_cal["Sim"].values)
    KGE_cal, r_cal, beta_cal, gamma_cal = kge(df_cal["Obs"].values, df_cal["Sim"].values)

    NSE_val = nse(df_val["Obs"].values, df_val["Sim"].values)
    RMSE_val = rmse(df_val["Obs"].values, df_val["Sim"].values)
    KGE_val, r_val, beta_val, gamma_val = kge(df_val["Obs"].values, df_val["Sim"].values)


### Plotting the output as figure

In [ ]:
    # Plot
    fig, ax = plt.subplots(figsize=(14,6))

    ax.plot(df["DateTime"], df["Obs"], color="black", linewidth=1.1, label="Q beobachtet")
    ax.plot(df["DateTime"], df["Sim"], color="royalblue", linewidth=1.1, label="Q simuliert")


    # Shading
    ax.axvspan(calib_start, calib_end, color="royalblue", alpha=0.13, label="Kalibrierung")
    ax.axvspan(valid_start, valid_end, color="orange", alpha=0.13, label="Validierung")


    # Labels
    ax.set_title("Beobachteter vs. simulierter Abfluss (Kalibrierung / Validierung)")
    ax.set_xlabel("Datum")
    ax.set_ylabel("Q [m³/s]")


    # Axis styling
    # Axis styling
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    ax.xaxis.set_minor_locator(mdates.MonthLocator())

    # Make major ticks stronger
    ax.tick_params(axis='x', which='major', length=4, width=0.9)
    ax.tick_params(axis='x', which='minor', length=2.5, width=0.5)


    # Grid
    ax.grid(True, which="major", axis="both", alpha=0.3)
    ax.grid(True, which="minor", axis="x", alpha=0.1)

    # Metrics box
    add_metrics_box(ax,
                NSE_cal, RMSE_cal, KGE_cal, r_cal, beta_cal, gamma_cal,
                NSE_val, RMSE_val, KGE_val, r_val, beta_val, gamma_val)
    plt.legend(loc="upper right")

    plt.tight_layout()

    plt.show()


In [ ]:
if __name__ == "__main__":
    main()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.offsetbox import AnchoredOffsetbox, TextArea, HPacker, VPacker
import matplotlib.patches as mpatches

# =============================================================================
# FILE PATHS
# =============================================================================
CALIBRATION_EXCEL = r"C:\Users\raah\Desktop\Aufgabe\TALSIM_NG\EZG\Ziegenrueck\Simulation_ZR\Calibration.xlsx"
J2K_CSV           = r"C:\Users\raah\Desktop\Aufgabe\TALSIM_NG\EZG\Ziegenrueck\Simulation_ZR\catchmentSimRunoff_qm.csv"

# =============================================================================
# TIME PERIODS
# =============================================================================

# Panel 1 - J2000 model
J2K_CALIB_START = "2012-11-01 00:00"
J2K_CALIB_END   = "2017-10-31 23:00"
J2K_VALID_START = "2017-11-01 00:00"
J2K_VALID_END   = "2022-10-31 23:00"

# Panel 2 - TALSIM model
TAL_CALIB_START = "2011-11-01 00:00"
TAL_CALIB_END   = "2017-10-31 23:00"
TAL_VALID_START = "2017-11-01 00:00"
TAL_VALID_END   = "2022-10-31 23:00"

# J2000 unit conversion: CSV is already in m3/s
J2K_UNIT_FACTOR = 1.0

# =============================================================================
# METRIC FUNCTIONS
# =============================================================================

def nse(q_obs, q_sim):
    mask = np.isfinite(q_obs) & np.isfinite(q_sim)
    q_obs, q_sim = q_obs[mask], q_sim[mask]
    if len(q_obs) == 0 or np.all(q_obs == q_obs[0]):
        return np.nan
    return 1 - np.sum((q_obs - q_sim) ** 2) / np.sum((q_obs - np.mean(q_obs)) ** 2)


def rmse(q_obs, q_sim):
    mask = np.isfinite(q_obs) & np.isfinite(q_sim)
    q_obs, q_sim = q_obs[mask], q_sim[mask]
    if len(q_obs) == 0:
        return np.nan
    return np.sqrt(np.mean((q_obs - q_sim) ** 2))


def kge(q_obs, q_sim):
    mask = np.isfinite(q_obs) & np.isfinite(q_sim)
    q_obs, q_sim = q_obs[mask], q_sim[mask]
    if len(q_obs) == 0:
        return np.nan, np.nan, np.nan, np.nan
    r = np.corrcoef(q_obs, q_sim)[0, 1] if np.std(q_obs) > 0 and np.std(q_sim) > 0 else np.nan
    mean_obs, mean_sim = np.mean(q_obs), np.mean(q_sim)
    std_obs,  std_sim  = np.std(q_obs),  np.std(q_sim)
    beta  = mean_sim / mean_obs if mean_obs != 0 else np.nan
    gamma = (std_sim / mean_sim) / (std_obs / mean_obs) if mean_obs != 0 and mean_sim != 0 and std_obs != 0 else np.nan
    if any(np.isnan(v) for v in [r, beta, gamma]):
        return np.nan, r, beta, gamma
    return 1 - np.sqrt((r - 1)**2 + (gamma - 1)**2 + (beta - 1)**2), r, beta, gamma


def compute_metrics(df_cal, df_val, obs_col, sim_col):
    def _m(df):
        o, s = df[obs_col].values, df[sim_col].values
        k, r, b, g = kge(o, s)
        return dict(NSE=nse(o, s), RMSE=rmse(o, s), KGE=k, r=r, beta=b, gamma=g)
    return _m(df_cal), _m(df_val)

# =============================================================================
# METRICS BOX - compact
# =============================================================================

def add_metrics_box(ax, cal, val, split_date):
    fmt = "{:<4s}{:>6.3f}"

    def col(header, m):
        lines = "\n".join([
            fmt.format("NSE :",  m["NSE"]),
            fmt.format("RMSE:", m["RMSE"]),
            fmt.format("KGE :",  m["KGE"]),
            fmt.format("r   :",    m["r"]),
            fmt.format("β   :",    m["beta"]),
            fmt.format("γ   :",    m["gamma"]),
        ])
        h = TextArea(header, textprops=dict(weight="bold", size=5, family="monospace"))
        b = TextArea(lines,  textprops=dict(size=5, family="monospace"))
        return VPacker(children=[h, b], align="left", pad=0, sep=2)

    hbox = HPacker(children=[col("Calibration", cal), col("Validation", val)],
                   align="top", pad=0, sep=8)
    # Placed with a placeholder anchor; repositioned below legend in plot_panel after legend is drawn
    ab = AnchoredOffsetbox(
        loc="upper right", child=hbox, frameon=True, borderpad=0.1,
        bbox_to_anchor=(1.7, 0.92),
        bbox_transform=ax.transAxes
    )
    ab.patch.set_boxstyle("square,pad=0.0")
    ab.patch.set_facecolor("white")
    ab.patch.set_alpha(0.7)
    ab.patch.set_edgecolor("0.7")
    ax.add_artist(ab)
    return ab

# =============================================================================
# VERTICAL DIVIDER - line only, no label
# =============================================================================

def add_divider(ax, split_date):
    ax.axvline(x=split_date, color="0.4", linewidth=0.5, linestyle="--", zorder=5)

# =============================================================================
# DATA LOADING
# =============================================================================

def load_excel(path):
    df = pd.read_excel(path)
    df["DateTime"] = pd.to_datetime(df["Zeit"], format="%d.%m.%Y %H:%M", errors="coerce")
    df = df.dropna(subset=["DateTime"])
    df = df.rename(columns={"With_Res": "Sim_TAL"})
    return df[["DateTime", "Obs", "Sim_TAL"]].set_index("DateTime").sort_index()


def load_j2k(path, unit_factor=1.0):
    for sep in ["\t", ";", ","]:
        df = pd.read_csv(path, sep=sep, decimal=".")
        df.columns = [c.strip() for c in df.columns]
        if len(df.columns) >= 2:
            break

    print(f"  J2K columns found: {df.columns.tolist()}")

    dt_col = next(
        (c for c in df.columns if any(k in c.lower() for k in ["date", "time", "zeit"])),
        df.columns[0]
    )
    run_col = next(
        (c for c in df.columns if any(k in c.lower() for k in ["runoff", "abfluss", "sim", "catchment"])),
        df.columns[1]
    )

    print(f"  Using datetime='{dt_col}', runoff='{run_col}'")

    df["DateTime"] = pd.to_datetime(df[dt_col], format="%d.%m.%Y %H:%M", errors="coerce")
    mask_na = df["DateTime"].isna()
    if mask_na.any():
        df.loc[mask_na, "DateTime"] = pd.to_datetime(df.loc[mask_na, dt_col], errors="coerce")

    df = df.dropna(subset=["DateTime"])
    df["Sim_J2K"] = pd.to_numeric(df[run_col], errors="coerce") * unit_factor
    return df[["DateTime", "Sim_J2K"]].set_index("DateTime").sort_index()

# =============================================================================
# SINGLE PANEL PLOT
# =============================================================================

def plot_panel(ax, df, obs_col, sim_col, sim_label,
               calib_start, calib_end, valid_start, valid_end,
               title, color_sim):

    full_start = min(calib_start, valid_start)
    full_end   = max(calib_end,   valid_end)
    df = df[(df.index >= full_start) & (df.index <= full_end)].copy()
    df = df.dropna(subset=[obs_col, sim_col])

    df_cal = df[(df.index >= calib_start) & (df.index <= calib_end)]
    df_val = df[(df.index >= valid_start) & (df.index <= valid_end)]

    cal_metrics, val_metrics = compute_metrics(df_cal, df_val, obs_col, sim_col)

    # sim first (bottom), obs on top
    ax.plot(df.index, df[sim_col], color=color_sim, linewidth=0.8,
            alpha=0.85, label=sim_label, zorder=2)
    ax.plot(df.index, df[obs_col], color="#1a1a2e", linewidth=0.8,
            label="Observed", zorder=3)

    # Period shading
    ax.axvspan(calib_start, calib_end, alpha=0.06, color="#2196F3", zorder=1)
    ax.axvspan(valid_start, valid_end, alpha=0.06, color="#FF9800", zorder=1)

    # Divider line only
    add_divider(ax, valid_start)

    # Metrics box (initially placed at upper right; repositioned below legend after draw)
    metrics_ab = add_metrics_box(ax, cal_metrics, val_metrics, valid_start)

    # Axes formatting
    ax.set_title(title, fontsize=6.5, fontweight="bold", pad=4, loc="left")
    ax.set_ylabel("Discharge [m3/s]", fontsize=6)
    ax.set_xlim(full_start, full_end)
    ax.yaxis.set_minor_locator(plt.AutoLocator())
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_minor_locator(mdates.MonthLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.tick_params(axis="x", which="major", labelsize=5.5)
    ax.tick_params(axis="x", which="minor", length=2, color="0.5", width=0.5)
    ax.tick_params(axis="y", labelsize=5.5)
    ax.grid(False, which="major")
    ax.grid(False, which="minor")
    
    # Legend - compact
    sim_line  = ax.get_lines()[0]
    obs_line  = ax.get_lines()[1]
    cal_patch = mpatches.Patch(facecolor="#2196F3", alpha=0.25, label="Calibration period")
    val_patch = mpatches.Patch(facecolor="#FF9800", alpha=0.25, label="Validation period")
    leg = ax.legend(handles=[obs_line, sim_line, cal_patch, val_patch],
              fontsize=5, loc="upper right", framealpha=0.8,
              borderpad=0.4, labelspacing=0.3, handlelength=1.5, handletextpad=0.4)

    # Reposition metrics box just below the legend
    ax.figure.canvas.draw()
    leg_bb   = leg.get_window_extent()
    # Convert legend bottom-left corner to axes fraction
    inv      = ax.transAxes.inverted()
    leg_x0, leg_y0 = inv.transform((leg_bb.x0, leg_bb.y0))
    leg_x1, _      = inv.transform((leg_bb.x1, leg_bb.y1))
    gap = 0.02   # small gap between legend bottom and metrics box top
    metrics_ab.set_bbox_to_anchor((leg_x1, leg_y0 - gap), transform=ax.transAxes)
    metrics_ab.loc = 1   # upper right anchor point of the box itself

# =============================================================================
# MAIN
# =============================================================================

def main():
    j2k_cs = pd.to_datetime(J2K_CALIB_START)
    j2k_ce = pd.to_datetime(J2K_CALIB_END)
    j2k_vs = pd.to_datetime(J2K_VALID_START)
    j2k_ve = pd.to_datetime(J2K_VALID_END)

    tal_cs = pd.to_datetime(TAL_CALIB_START)
    tal_ce = pd.to_datetime(TAL_CALIB_END)
    tal_vs = pd.to_datetime(TAL_VALID_START)
    tal_ve = pd.to_datetime(TAL_VALID_END)

    print("Loading Excel ...")
    df_excel = load_excel(CALIBRATION_EXCEL)

    print("Loading J2000 CSV ...")
    df_j2k = load_j2k(J2K_CSV, unit_factor=J2K_UNIT_FACTOR)

    df_j2k_merged = df_excel[["Obs"]].join(df_j2k, how="inner").dropna()
    df_tal        = df_excel[["Obs", "Sim_TAL"]].dropna()

    fig, (ax1, ax2) = plt.subplots(
        2, 1, figsize=(6.30, 6.20), sharex=False,
        gridspec_kw={"hspace": 0.28}
    )

    plot_panel(
        ax=ax1, df=df_j2k_merged, obs_col="Obs", sim_col="Sim_J2K",
        sim_label="Simulated J2000",
        calib_start=j2k_cs, calib_end=j2k_ce,
        valid_start=j2k_vs, valid_end=j2k_ve,
        title="a) J2000-Model: Observed vs. Simulated",
        color_sim="#e63946",
    )

    plot_panel(
        ax=ax2, df=df_tal, obs_col="Obs", sim_col="Sim_TAL",
        sim_label="Simulated TALSIM",
        calib_start=tal_cs, calib_end=tal_ce,
        valid_start=tal_vs, valid_end=tal_ve,
        title="b) TALSIM-Model: Observed vs. Simulated",
        color_sim="#2a9d8f",
    )

    ax2.set_xlabel("Time", fontsize=6)

    fig.tight_layout()
    fig.subplots_adjust(hspace=0.28)
    plt.savefig("hydro_comparison.png", dpi=300, bbox_inches="tight")
    print("Saved -> hydro_comparison.png")
    plt.show()


if __name__ == "__main__":
    main()